In [1]:
import pandas as pd
import numpy as np
import joblib

# Load trained XGBoost pipeline
model = joblib.load("best_churn_model.pkl")

# Load optimized threshold
threshold = joblib.load("churn_threshold.pkl")

print("Model loaded successfully!")
print("Threshold:", threshold)


Model loaded successfully!
Threshold: 0.35


In [2]:
customer = pd.DataFrame([{
    'CreditScore': 650,
    'Geography': 'Germany',
    'Gender': 'Female',
    'Age': 45,
    'Tenure': 3,
    'Balance': 120000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 0,
    'EstimatedSalary': 100000
}])

customer

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,650,Germany,Female,45,3,120000,2,1,0,100000


In [3]:
# Predict churn probability
churn_probability = model.predict_proba(customer)[:, 1][0]

# Apply optimized threshold
prediction = 1 if churn_probability >= threshold else 0

# Display results
print("Churn Probability:", round(churn_probability * 100, 2), "%")

if prediction == 1:
    print("Prediction: Customer is likely to LEAVE.")
else:
    print("Prediction: Customer is likely to STAY.")

Churn Probability: 42.07 %
Prediction: Customer is likely to LEAVE.


In [6]:
def predict_churn(customer_data):

    customer_df = pd.DataFrame([customer_data])

    probability = float(model.predict_proba(customer_df)[0][1])

    prediction = 1 if probability >= threshold else 0

    result = {
        'Churn Probability (%)': round(probability * 100, 2),
        'Prediction': 'Churn' if prediction == 1 else 'No Churn'
    }

    return result

In [7]:
new_customer = {
    'CreditScore': 720,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 30,
    'Tenure': 8,
    'Balance': 0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 85000
}

result = predict_churn(new_customer)

print(result)

{'Churn Probability (%)': 0.96, 'Prediction': 'No Churn'}


# Final Project Conclusion

## Project Summary

Developed a Machine Learning-based Customer Churn Prediction system using the Bank Customer Churn dataset.

The objective was to identify customers who are likely to leave the bank so that the bank can take preventive actions to improve customer retention.

## Models Implemented

Seven classification algorithms were trained and compared:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine (SVM)
5. K-Nearest Neighbors (KNN)
6. Naive Bayes
7. XGBoost

## Model Selection

XGBoost was selected based on its overall performance, particularly its F1-score, ROC-AUC, and accuracy.

Hyperparameter tuning was performed using RandomizedSearchCV with 3-fold Stratified Cross-Validation.

## Final Model Performance

| Metric | Original Threshold (0.50) | Adjusted Threshold (0.35) |
|---|---:|---:|
| Accuracy | 86.55% | 85.65% |
| Precision | 78.75% | 65.87% |
| Recall | 46.44% | 61.18% |
| F1-score | 58.42% | 63.44% |

## Threshold Optimization

The classification threshold was reduced from 0.50 to 0.35 to improve the identification of potential churners.

This increased recall and F1-score, allowing the model to identify more customers who may leave the bank.

## Key Business Insights

- Customers from Germany showed the highest churn rate.
- Female customers had a higher churn rate than male customers.
- Older customers showed a greater tendency to leave.
- Inactive members had a higher churn rate.
- Customers with three or four products showed high churn rates, although these groups had relatively few records.

## Limitations

- The model may miss some customers who actually leave.
- Lowering the classification threshold increases false-positive predictions.
- Predictions are based on historical data and may not reflect future customer behavior perfectly.
- The model's probability estimates are not guarantees of actual customer behavior.

## Final Conclusion

This project demonstrates an end-to-end Machine Learning workflow, including data cleaning, exploratory data analysis, feature engineering, model comparison, hyperparameter tuning, evaluation, threshold optimization, and prediction on new customer records.

The final XGBoost pipeline provides a practical approach to identifying potential customer churn and supporting data-driven customer retention strategies.

In [8]:
import joblib

model = joblib.load("best_churn_model.pkl")

print("MODEL STRUCTURE:")
print(model)

print("\nPIPELINE STEPS:")
for name, step in model.named_steps.items():
    print(name, ":", type(step))

MODEL STRUCTURE:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['CreditScore', 'Age',
                                                   'Tenure', 'Balance',
                                                   'NumOfProducts', 'HasCrCard',
                                                   'IsActiveMember',
                                                   'EstimatedSalary']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Geography', 'Gender'])])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_polic

In [10]:
import joblib
import pandas as pd

model = joblib.load("best_churn_model.pkl")

preprocessor = model.named_steps["preprocessor"]
xgb_model = model.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
importances = xgb_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df["Feature"] = (
    importance_df["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

importance_df = importance_df.sort_values(
    by="Importance", ascending=False
).reset_index(drop=True)

display(importance_df)

,Feature,Importance
0,Age,0.196103
1,NumOfProducts,0.188853
2,IsActiveMember,0.174236
3,Geography_Germany,0.122421
4,Gender_Female,0.084096
5,Balance,0.058939
6,Geography_France,0.048416
7,HasCrCard,0.028801
8,CreditScore,0.028035
9,EstimatedSalary,0.023964
